Derived from 

**DarkCapPy Template**

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import interpolate
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.style
mpl.rcParams.update(mpl.rcParamsDefault)
%matplotlib inline
from datetime import datetime

from DarkCapPy import *
import DarkCapPy.DarkPhoton as DP

import time
import seaborn as sns

###################
# Define File Paths
###################
# These paths are specific to the preset folders in Temp;ate_Caluclation
def sommerfeldPath(file):
    path = 'Sommerfeld/' + file
    return path

def branchPath(file):
    path = 'Branching_Ratio/' + file
    return path

def signalPath(file):
    path = 'Signal/' + file
    return path
    
def signalBackupPath(file):
    path = 'Signal/Signal_Backups/' + file
    return path

print ('Complete')

# Check the parameters for the planetary body

I run this just to make sure I'm working with what I think I am. 

[Earth for reference](https://en.wikipedia.org/wiki/Earth)

In [ ]:
# Check to make sure we are using the correct versions of thing

# The Earth is about 6000 km in radius

print(f'Planet mass:   {DP.Planet_Mass} grams')
print(f'Planet radius: {DP.Planet_Radius} cm')
print(f'Planet radius: {DP.Planet_Radius/1e5} km')

# Get the kappa values

You should have calculated these previously and saved them to some file. 

In [ ]:
# For testing
#infilename = 'KAPPAS_FOR_TESTING.parquet'
infilename = 'KAPPAS_10_1000000_varying_steps.parquet'

df_kappa = pd.read_parquet(infilename)

kappa_tag = infilename.split('.parquet')[0]
print(f'kappa tag: {kappa_tag}\n')

# Grab a single kappa value for demonstration
mx_test = df_kappa['mx'].iloc[0]

filter = df_kappa['mx'] == mx_test

kappa0 = df_kappa[filter]['kappa0'].values[0]

print (f'mx: {mx_test}    Kappa_0: {kappa0}')
print()

df_kappa

# Diagnostics

Peform a calculation for a single mass point, just to check things



In [ ]:

ma = 0.30
epsilon = 1e-8
alpha = 1/137

# Grab a point we know is in the file
mx = df_kappa['mx'].iloc[0]


filter = df_kappa['mx'] == mx
kappa0 = df_kappa[filter]['kappa0'].values[0]
alphax = DP.alphaTherm(mx, ma)
cap2 = DP.cCapQuick(mx,ma,epsilon,alphax,kappa0)

planet_lifetime = DP.Planet_Life
v0 = DP.v0func(mx)
sommerfeld = DP.thermAvgSommerfeld(mx,ma,alphax)
sigma = DP.sigmaVtree(mx,ma,alphax)
ann = DP.cAnn(mx,sigma,sommerfeld)
tau = DP.tau(cap2,ann)
gammaAnn = DP.gammaAnn(cap2,ann)
L = DP.decayLength(mx,ma,epsilon,1)
Edecay = DP.epsilonDecay(L)
n_particles = cap2*planet_lifetime

print ('Kappa_0              :', kappa0)
print ('Capture rate         :', cap2)
print ('Time to accum (s)    :', planet_lifetime)
print ('# accum              :', n_particles)

print ('DM velocity          :', v0)
print ('DM velocity (with c) :', v0*3e8)

print ('Therm Avg Sommerfeld :', sommerfeld)
print ('Sigma V              :', sigma)
print ('Annihilation         :', ann)
print ('EQ Time              :', tau)
print ('Gamma_ann (#/sec?)   :', gammaAnn)
print ('Decay Length (cm)    :', L)
print ('Decay Length (km)    :', L/1e5)
print ('Epsilon_decay        :', Edecay)

print()



# Calculate rates

We first need the branching fraction of the dark photon to muons or electrons, which we extracted from 
this paper, and saved the points in a file. 

In [ ]:
#epsilons
df_br_muon = pd.read_csv('Branching_Ratio/brto_muon_extracted_from_paper.csv')
df_br_muon

df_br_electron = pd.read_csv('Branching_Ratio/brtoe.csv')
df_br_electron


#df_br_muon.plot(x='x (GeV)', y='y (branching ratio)', figsize=(12,4))

xe = df_br_electron['mA[GeV]']
ye = df_br_electron['BR']

xmu = df_br_muon['x (GeV)']
ymu = df_br_muon['y (branching ratio)']

plt.plot(xe, ye, label='BR(electron)')
plt.plot(xmu, ymu, label='BR(muon)')

plt.xscale('log')

bre_interp = interpolate.interp1d(xe, ye)
brmu_interp = interpolate.interp1d(xmu, ymu)


xptse = np.linspace(0.1, 10, 1000)
xptsmu = np.linspace(0.221, 10, 1000)

yptse = bre_interp(xptse)
yptsmu = brmu_interp(xptsmu)

plt.plot(xptse, yptse, label='interp(electron)')
plt.plot(xptsmu, yptsmu, label='interp(electron)')

plt.legend()

Now generate the rates. First set up the parameters we'll be using to calculate the different rates.

In [ ]:
# If you want to read in a different kappa file at this stage, 
# here is the code
#'''
#infilename = 'KAPPAS_FOR_TESTING.parquet'
#infilename = 'KAPPAS_10_100_1000_10000_100000.parquet'
infilename = 'KAPPAS_10_1000000_varying_steps.parquet'

df_kappa = pd.read_parquet(infilename)
kappa_tag = infilename.split('.parquet')[0]
print(f'kappa tag: {kappa_tag}\n')
#'''

# Fine structure constant, natch
alpha = 1./137

# Planet lifetime
planet_lifetime = DP.Planet_Life

# Age of Earth. Time to accumulate dark matter
# In the code this, is the same as DP.Planet_Life
tauCross = DP.tauCross
print(f"tauCross: {tauCross:.2e} seconds")
print(f"tauCross: {tauCross/3.1e7:.2e} years")
print()

# Detector live time
live_time = 1 # years

#####################################################################################
# For the full range
#mxs = df_kappa['mx'].values
# Could choose to do this for a subset of mx values
#mxs = [10, 100, 1000, 10000, 100000]
#mxs = [10, 100]
mxs = [1000]
#mxs = [10000, 100000]


print(f"Running over {len(mxs)} values of m_X from {min(mxs)} to {max(mxs)}\n")
#####################################################################################


#####################################################################################
# We will use this to name the output file
#mass_tag = kappa_tag
#mass_tag = 'SOME_OTHER_DESCRIPTIVE_TERM'
#mass_tag = f'{kappa_tag}_fine_grain_epsilon_and_mas'
#mass_tag = f'10_100_1000_10000_100000_fine_grain_epsilon_and_mas'
mass_tag = f'1000_fine_grain_epsilon_and_mas'
#mass_tag = f'10_100_fine_grain_epsilon_and_mas'
#mass_tag = f'10000_100000_fine_grain_epsilon_and_mas'

#mass_tag = f'{kappa_tag}_coarse_grain_epsilon_and_mas'

print(f'mass tag: {mass_tag}\n')
#####################################################################################

#####################################################################################
# mA (dark photon) mass points
#
#mas = np.linspace(0.1, 1.0, 100).tolist()

####################################################
# Full range for Fig 3-type plots
mas = np.linspace(0.01, 0.1, 1000).tolist()
mas += np.linspace(0.1, 1.0, 1000).tolist()
mas += np.linspace(1.0, 10.0, 1000).tolist()
####################################################

####################################################
# Focus on above dimuon threshold
#mas = np.linspace(0.22, 1.0, 79).tolist()

# For testing
#mas = [0.25, 0.5]

print(f"Running over {len(mas)} values of m_a from {min(mas)} to {max(mas)}\n")
#####################################################################################

#####################################################################################
# epsilon values
#
#
# For testing
#epsilons = [1e-9, 1e-8, 1e-7]

# Finer granularity
#epsilons  = np.linspace(1e-9, 1e-6, 1000).tolist() 

#'''
# Normal epsilon points
epsilons  = np.linspace(1e-11, 9e-11, 99).tolist() 
epsilons += np.linspace(1e-10, 9e-10, 99).tolist() 
epsilons += np.linspace(1e-9,  9e-9,  99).tolist()
epsilons += np.linspace(1e-8,  9e-8,  99).tolist()
epsilons += np.linspace(1e-7,  9e-7,  99).tolist()
#epsilons += np.linspace(1e-6, 9e-6, 9).tolist()
#'''

'''
# Coarse epsilon points
epsilons  = np.linspace(1e-11, 9e-11, 9).tolist() 
epsilons += np.linspace(1e-10, 9e-10, 9).tolist() 
epsilons += np.linspace(1e-9,  9e-9,  9).tolist()
epsilons += np.linspace(1e-8,  9e-8,  9).tolist()
epsilons += np.linspace(1e-7,  9e-7,  9).tolist()
#epsilons += np.linspace(1e-6, 9e-6, 9).tolist()
'''

print(f"Running over {len(epsilons)} values of epsilon from {min(epsilons)} to {max(epsilons)}\n")
#####################################################################################

print()

#print("ma values: ")
#print(mas)
#print()
#print("epsilon values: ")
#print(epsilons)

In [ ]:
# Prepare a dictionary to store everything 
# We'll convert this to a dataframe afterwards
dict_results = {}
dict_results['mx'] = []
dict_results['ma'] = []
dict_results['kappa0'] = []
dict_results['alphax'] = []
dict_results['alpha_therm_or_max'] = []
dict_results['BR'] = []
dict_results['epsilon'] = []
dict_results['rate_1yr'] = []
dict_results['rate_CMS_1yr'] = []
dict_results['livetime_years'] = []
dict_results['depth_scale'] = []
dict_results['angular_acceptance'] = []
dict_results['final_state_particles'] = []

dict_results['ann'] = []
dict_results['gammaAnn'] = []
dict_results['L'] = []
dict_results['Edecay'] = []
dict_results['n_particles'] = []


print("Starting the calculations...\n")

for mx in mxs:

    start = time.time()

    # Extract the kappa0 value
    filter = df_kappa['mx'] == mx
    kappa0 = df_kappa[filter]['kappa0'].values[0]

    v0 = DP.v0func(mx)

    for ma in mas:
        # This would give us an error in the alphaThermApprox calculation
        if ma == mx:
            continue

        
        # Calculate alphax values and we'll calculate rates for each 
        alphax_thermal_relics = DP.alphaTherm(mx, ma)
        alphax_max = 0.17 * (mx/1000)**1.61

    
        for alphax,therm_or_max in zip([alphax_thermal_relics, alphax_max], ['THERMAL', 'MAX']):

            sommerfeld = DP.thermAvgSommerfeld(mx,ma,alphax)
            
            sigma = DP.sigmaVtree(mx,ma,alphax)
            
            ann = DP.cAnn(mx,sigma,sommerfeld)

            # Extract the branching fractions for both muons and electrons            
            br_electrons = bre_interp(ma)
            
            br_muons = 0
            if ma>0.22:
                br_muons = brmu_interp(ma)
            
            #####################################################################
            # WE KNOW THIS IS NOT CORRECT BUT THIS WAS USED IN EARLIER CALCULATIONS
            # FOR CMS SO IT'S USEFUL TO SAVE THIS INFO FOR COMPARISONS
            #####################################################################            
            # Scale
            # Do a rough area scale
            angular_acceptance_scale = (20**2)/(1000**2)
            
            # Do a rough depth scale, based on the energy loss in rock
            depth_scale = 1
            if mx == 10:
                depth_scale = 10 / 1000
            elif mx == 100:
                depth_scale = 100 / 1000
            else:
                depth_scale = 1000/1000
            #####################################################################

            # Loop over muons or electrons for the final state
            for branching_fraction_to_final_state_particles, final_state_particles in zip([br_muons, br_electrons],['muons', 'electrons']):
        
                for epsilon in epsilons:
        
                    if branching_fraction_to_final_state_particles > 0:

                        cap1 = DP.cCapQuick(mx, ma, epsilon, alphax, kappa0)                                   
                        n_particles = cap1*planet_lifetime
                        
                        tau = DP.tau(cap1, ann)
                        
                        gammaAnn = DP.gammaAnn(cap1,ann)
                                    
                        L = DP.decayLength(mx,ma,epsilon,branching_fraction_to_final_state_particles)
                        
                        Edecay = DP.epsilonDecay(L)
                        
                        # Rate
                        signal = DP.iceCubeSignal(gammaAnn,Edecay,DP.yr2s(live_time))
                    
                    else:
                        tau = 0
                        gammaAnn = 0
                        L = 0
                        Edecay = 0
                        signal = 0
                        cap1 = 0
                        n_particles = 0
                        tau = 0
                        sigma = 0
                        
                                    
                    # AGAIN, THIS IS NOT CORRECT. WE'LL SCALE THE ICECUBE RATE
                    # USING EARTHSHINE CALCULATIONS
                    signal_CMS = signal*angular_acceptance_scale*depth_scale
                    
                    # For some reason this was getting saved as an object at times. 
                    br = branching_fraction_to_final_state_particles
                    try:
                        float(br)
                        branching_fraction_to_final_state_particles = float(branching_fraction_to_final_state_particles)
                    except ValueError:
                        print(br)
                        print(f"mx: {mx:5d}  ma: {ma:4.3f}   epsilon: {epsilon:.2e}     kappa0: {kappa0:.2e}   # signal: {signal:10.2f}  # CMS signal: {signal_CMS:.2f}")
        
                        print("Not a float")
                    
                    dict_results['mx'].append(mx)
                    dict_results['ma'].append(ma)
                    dict_results['kappa0'].append(kappa0)
                    dict_results['alphax'].append(alphax)
                    dict_results['BR'].append(branching_fraction_to_final_state_particles)
                    dict_results['epsilon'].append(epsilon)
                    dict_results['rate_1yr'].append(signal)
                    dict_results['rate_CMS_1yr'].append(signal_CMS)
                    dict_results['livetime_years'].append(live_time)
                    dict_results['alpha_therm_or_max'].append(therm_or_max)
                    dict_results['depth_scale'].append(depth_scale)
                    dict_results['angular_acceptance'].append(angular_acceptance_scale)
                    dict_results['final_state_particles'].append(final_state_particles)

                    dict_results['ann'].append(ann)
                    dict_results['gammaAnn'].append(gammaAnn)
                    dict_results['L'].append(L)
                    dict_results['Edecay'].append(Edecay)
                    dict_results['n_particles'].append(n_particles)

    time_to_run = time.time() - start
    print(f"mx: {mx}    kappa0: {kappa0:.2e}     time to run: {time_to_run:.2f} s")

###################################################################

print(f"Converting from a dictionary to a dataframe")
start = time.time()
df_results = pd.DataFrame.from_dict(dict_results)
time_to_run = time.time() - start
print(f"Time to convert: {time_to_run:.2f} s")

filename = f'rates_muons_electrons_both_alphas_{mass_tag}.parquet'

print(f'Saving file as {filename}')
start = time.time()
df_results.to_parquet(filename)
time_to_run = time.time() - start
print(f"Time to write file: {time_to_run:.2f} s")


In [ ]:
df_results

In [ ]:
df_results['rate_1yr'].max()

In [ ]:
del df_results